In [1]:
import numpy as np
import math
import random
import sympy
from sympy import poly as _poly, Poly
from sympy.abc import x
from sympy.core.symbol import symbols
from dataclasses import dataclass

from vm import VM
from merkle import MerkleTree, verify_decommitment

# 1. Public Parameters
Both verifier and prover need to agree on the following public parameters. Everyone can see it.

- A large enough finite field $\mathbb{F}$ (and it's modulo $M$ and primitive root $omega$), all operations are in this field
- The CI(computational Integrity) statement, aka the VM
- $\rho$, the multiple by which the domain is enlarged. Larger $\rho$ will increase the amount of calculation on the prover side but decrease the round needed in FRI
- The maximum degree $d$ of $p(x)$, which is derive from the structure of CI

## 1.1 The Finite Field

In [2]:
M = 2**2**4 + 1 # the largest known fermat number
print(f'{M = }')
F = sympy.FiniteField(M)

M = 65537


In [3]:
def poly(f):
    return _poly(f, x, domain=F)

In [4]:
def prime_factor(M: int):
    factors = set()
    if M % 2 == 0:
        factors.add(2)
        while M % 2 == 0:
            M //= 2
    for i in range(3, int(np.sqrt(M)) + 1, 2):
        if M % i == 0:
            factors.add(i)
            while M % i == 0:
                M //= i
    if M > 1:
        factors.add(M)
    return list(factors)

In [5]:
prime_factors = prime_factor(M-1)
omega = 0
for a in range(2, M):
    for p in prime_factors:
        c = M//p
        if a**(M//p) % M == 1:
            break
    else:
        omega = a
        break

print(f'{omega = }')

omega = 3


In [6]:
rho = 4

## 1.2 The CI statement

In [7]:
r1_curr, r1_next = symbols("r1_curr r1_next")
r2_curr, r2_next = symbols("r2_curr r2_next")
r1_first, r2_first = symbols("r1_first r2_first")
inp, set_op, swap_op, add_op, mul_op, nop_op = symbols("input set swap add mul nop")

all_gens = (
    r1_curr,
    r1_next,
    r2_curr,
    r2_next,
    r1_first,
    r2_first,
    inp,
    set_op,
    swap_op,
    add_op,
    mul_op,
    nop_op,
)

constraints = [
    # u_all
    {"expr": set_op**2 - set_op, "domain": "all"},
    {"expr": swap_op**2 - swap_op, "domain": "all"},
    {"expr": add_op**2 - add_op, "domain": "all"},
    {"expr": mul_op**2 - mul_op, "domain": "all"},
    {"expr": nop_op**2 - nop_op, "domain": "all"},
    {"expr": set_op + swap_op + add_op + mul_op + nop_op - 1, "domain": "all"},
    {"expr": (1 - set_op) * inp, "domain": "all"},
    # u_trans
    {"expr": set_op * (r1_next - r1_curr), "domain": "trans"},
    {"expr": set_op * (r2_next - inp), "domain": "trans"},
    {"expr": swap_op * (r1_next - r2_curr), "domain": "trans"},
    {"expr": swap_op * (r2_next - r1_curr), "domain": "trans"},
    {"expr": add_op * (r1_next - (r1_curr + r2_curr)), "domain": "trans"},
    {"expr": add_op * (r2_next - r2_curr), "domain": "trans"},
    {"expr": mul_op * (r1_next - (r1_curr * r2_curr)), "domain": "trans"},
    {"expr": mul_op * (r2_next - r2_curr), "domain": "trans"},
    {"expr": nop_op * (r1_next - r1_curr), "domain": "trans"},
    {"expr": nop_op * (r2_next - r2_curr), "domain": "trans"},
    # u_first
    {"expr": r1_curr - r1_first, "domain": "first"},
    {"expr": r2_curr - r2_first, "domain": "first"},
]

len(constraints)

19

In [8]:
for c in constraints:
    print(c['expr'])

set**2 - set
swap**2 - swap
add**2 - add
mul**2 - mul
nop**2 - nop
add + mul + nop + set + swap - 1
input*(1 - set)
set*(-r1_curr + r1_next)
set*(-input + r2_next)
swap*(r1_next - r2_curr)
swap*(-r1_curr + r2_next)
add*(-r1_curr + r1_next - r2_curr)
add*(-r2_curr + r2_next)
mul*(-r1_curr*r2_curr + r1_next)
mul*(-r2_curr + r2_next)
nop*(-r1_curr + r1_next)
nop*(-r2_curr + r2_next)
r1_curr - r1_first
r2_curr - r2_first


In [9]:
def make_u(G: list[int]):
    u_all = poly(1)
    for gg in G:
        u_all = u_all * poly(x-gg)
    
    u_trans = poly(1)
    for gg in G[:-1]:
        u_trans = u_trans * poly(x-gg)
    
    u_first = poly(x - G[0])
    u_last = poly(x - G[-1])
    
    u = {
        'all': u_all,
        'trans': u_trans,
        'first': u_first,
    }

    return u

# 2. Executation

In [10]:
@dataclass(frozen=True)
class PolyPoint:
    poly: str
    index: int
    value: int
    proof: list[str]

In [11]:
@dataclass
class PolyCollection:
    L: list(int)
    polys: dict[str, MerkleTree]

    def insert(self, name: str, poly: Poly):
        """
        evaluate a poly and put in collection
        """
        data = [poly(x) for x in self.L]
        merkle = MerkleTree(data)
        self.polys[name] = merkle
    
    def add(self, name: str, merkle: MerkleTree):
        """
        add a MerkleTree of an evaluated poly in the collection
        """
        self.polys[name] = merkle

    def merge(self, other: PolyCollection):
        """
        merge two collections
        """
        self.polys = self.polys | other.polys

    def get(self, name: str, z: int):
        """
        get value and proof of a point in a polynomial
        """
        if name not in self.polys:
            raise Exception(f'{name} not in this collection')
        if z not in self.L:
            raise Excpetion(f'{z} is out of L')
        index = self.L.index(z)
        value = self.polys[name].data[index]
        proof = self.polys[name].get_authentication_path(index)

        return PolyPoint(name, index, value, proof)

    def proof(self):
        """
        get proof(collection of roots)
        """
        polys = {name: poly.root for name, poly in self.polys.items()}
        return PolyProof(polys, None)
        

@dataclass
class PolyProof:
    polys: dict[str, list[str]]
    __get: Callable[[str, int], PolyPoint] | None

    def add(self, name, poly):
        self.polys[name] = poly
    
    def merge(self, other: PolyProof):
        self.polys = self.polys | other.polys

    def set_get(self, get: Callable[[str, int], PolyPoint]):
        self.__get = get

    def get(self, name: str, z: int):
        return self.__get(name, z) 
        
    def verify(self, p: PolyPoint):
        if not verify_decommitment(p.index, p.value, p.proof, self.polys[p.poly]):
            raise Exception(f'f_{p.poly}[{p.index}] != {p.value}')
        return F(p.value)

In [12]:
def interpolation(xs: list, ys: list):    
    assert len(xs) == len(ys)
    N = len(xs)

    f = poly(0)

    for i in range(N):
        li = poly(1)
        for j in range(N):
            if i == j:
                continue
            term = poly(x - xs[j] ) * F(xs[i] - xs[j])**-1
            li = li * term
        f = f + F(ys[i]) * li

    assert f.degree() < N
    return f

In [13]:
class Prover:
    def __init__(self, instructions):
        # instruction -> trace
        vm = VM()
        T = vm.exec(instructions) # trace
        N = vm.N                  # the length of trace
        d_bar = 3*(N-1)-N

        print('step', 'r1', 'r2', 'input', 'set', 'swap', 'add', 'mul', 'nop', sep='\t')
        for i, step in enumerate(T):
            print(i, end='\t')
            for t in step:
                print(t, end='\t')
            print()
        print()
        
        # choose G based on the length of trace
        g = F(omega**((M-1)//N)) # the generator of G
        G = [F(g**i) for i in range(N)]
        print(g)
        print([int(i) for i in G])

        # interpolation trace into f
        fk = {
            'r1_curr': interpolation(G, [row[0] for row in T]),
            'r2_curr': interpolation(G, [row[1] for row in T]),
            'input': interpolation(G, [row[2] for row in T]),
            'set': interpolation(G, [row[3] for row in T]),
            'swap': interpolation(G, [row[4] for row in T]),
            'add': interpolation(G, [row[5] for row in T]),
            'mul': interpolation(G, [row[6] for row in T]),
            'nop': interpolation(G, [row[7] for row in T]),
        }

        # choose larger subgroup L
        h = F(omega**((M-1)//(N*rho))) # the generator of L
        L = [F(h**i) for i in range(N*rho)]

        # apply f to constraints
        f_sym = {
            r1_curr: fk['r1_curr'].as_expr(),
            r2_curr: fk['r2_curr'].as_expr(),
            r1_next: fk['r1_curr'].as_expr().subs(x, g * x),
            r2_next: fk['r2_curr'].as_expr().subs(x, g * x),
            r1_first: T[0][0],
            r2_first: T[0][1],
            inp: fk['input'].as_expr(),
            set_op: fk['set'].as_expr(),
            swap_op: fk['swap'].as_expr(),
            add_op: fk['add'].as_expr(),
            mul_op: fk['mul'].as_expr(),
            nop_op: fk['nop'].as_expr(),
        }

        ck = [{
            "poly": poly(c['expr'].subs(f_sym)), 
            "domain": c['domain'],
        } for c in constraints]

        # the vanishing function
        u = make_u(G)
        pc = PolyCollection(L, {})

        for k in fk:
            pc.insert(k, fk[k])
            
        self.__N = N
        self.__G = G
        self.__L = L
        self.__fk = fk
        self.__ck = ck
        self.__u = u
        self.__pc = pc
        self.__d_bar = d_bar


    def get(self, name: str, z: int):
        return self.__pc.get(name, z)
        
    def commit_proof(self):
        return self.__N, self.__pc.proof()

    def prepare_FRI(self, betak):
        """
        get beta_k and construct p
        """

        numerator = {
            'all': poly(0),
            'trans': poly(0),
            'first': poly(0),
        }
        
        # 3. 第一步：純粹做分子的隨機線性組合（只加總，絕對不提前做除法）
        for c, beta in zip(self.__ck, betak):
            numerator[c['domain']] += beta * c['poly']

        p0 = poly(0)
        r = {}

        for k, n in numerator.items():
            pk, r[k] = n.div(self.__u[k])
            if r[k] != poly(0):
                raise Exception(f'{r_[k]} = {r[k]}')
            p0 = p0 + pk
            
        print(f'{p0 = }')

        data = [p0(x) for x in self.__L]

        merkle = MerkleTree(data)
        self.__pc.add('p0', data)
        self.__p0 = p0

        return merkle.root

    def request_fri(self, zs, alpha):
        """
        input a list of z and alpha, this function will 
        fold p0 with alpha at each z, then return the proof

        return values:
            p0zs: []p(z), for verifier to check if c(z) = p(z) * u(z)
            queries: [][](p_k(z), p_k(-z), p_(k+1)(z^2)
            polyProofs: []PolyProof
        """

        p0zs = []
        queries = []
        polyProofs = []
        
        # e.g. d_bar = 14
        # we need to do 4 layers because 2^4 > 14
        l = self.__d_bar.bit_length()
        for z in zs:
            p0zs.append(self.__p0(z))
            p = [poly(0)] * (l+1)
            p[0] = self.__p0
            query = []
            polyCollection = PolyCollection(self.__L, {})
            polyCollection.insert('p0', self.__p0)

            for i in range(l):
                zz = z**(2**i)
                p[i+1] = FRI_fold(p[i], alpha[i])
                polyCollection.insert(f'p{i+1}', p[i+1])
                query.append((
                    polyCollection.get(f'p{i}', zz),
                    polyCollection.get(f'p{i}', -zz),
                    polyCollection.get(f'p{i+1}', zz**2),
                ))

            queries.append(query)
            polyProofs.append(polyCollection.proof())
                    
        return p0zs, queries, polyProofs

def FRI_fold(p0, alpha0):
    coff0 = p0.all_coeffs()[::-1]
    g0_coff = [coff0[i] for i in range(0, len(coff0), 2)]
    h0_coff = [coff0[i] for i in range(1, len(coff0), 2)]
    g0 = poly(sum(c * (x**i) for i, c in enumerate(g0_coff)))
    h0 = poly(sum(c * (x**i) for i, c in enumerate(h0_coff)))
    p1 = g0 + alpha0 * h0
    return p1

In [14]:
# p = Prover([
#     # (1+2)*4
#     'set 1',
#     'swap',
#     'set 2',
#     'add',
#     'set 4',
#     'mul',
# ])

p = Prover([
    # (1+2)*4+6
    'set 1',
    'swap',
    'set 2',
    'add',
    'set 4',
    'mul',
    'set 6',
    'add'
])


N, proof = p.commit_proof()
proof.set_get(p.get)
N, proof

step	r1	r2	input	set	swap	add	mul	nop
0	0	0	1	1	0	0	0	0	
1	0	1	0	0	1	0	0	0	
2	1	0	2	1	0	0	0	0	
3	1	2	0	0	0	1	0	0	
4	3	2	4	1	0	0	0	0	
5	3	4	0	0	0	0	1	0	
6	12	4	6	1	0	0	0	0	
7	12	6	0	0	0	1	0	0	
8	18	6	0	0	0	0	0	1	
9	18	6	0	0	0	0	0	1	
10	18	6	0	0	0	0	0	1	
11	18	6	0	0	0	0	0	1	
12	18	6	0	0	0	0	0	1	
13	18	6	0	0	0	0	0	1	
14	18	6	0	0	0	0	0	1	
15	18	6	0	0	0	0	0	1	

64 mod 65537
[1, 64, 4096, 65533, 65281, 49153, 16, 1024, 65536, 65473, 61441, 4, 256, 16384, 65521, 64513]


(16,
 PolyProof(polys={'r1_curr': 'af4b32c35cfcf8565576f6352f3cddf6e16aba6f5ea155a28ee0712f7c7b5dbd', 'r2_curr': '0dfb528337610b2dc7def0b5794a92a382ecfcb6dd54222d357504ec91b5a3e1', 'input': '3ff0d0cb370009db9486bd909c719c2dbdd671c09e775d054586b290fff74b02', 'set': '3776a76b0ca3e87c60432cdff05bb1cdaace43c52a4f46f5f87c784e4b7e5913', 'swap': '28bc66600332337141608ba67f193558962dabad8eba74d4886f901fcf99ade5', 'add': '423fcf6d54c7b0eddd906e634b0e54440631508133d2614a7086e50f32d63cbe', 'mul': 'a3485ec6b5db348aab8e3f97c72ff8b1948ae1db7b5ca2422b2a58d742d83218', 'nop': '07f6b3a4ca0605946d62dbf31fc0a90eca07d1ac5a23b2a027d00d7fcca8619e'}, _PolyProof__get=<bound method Prover.get of <__main__.Prover object at 0x7f74f6f9e3c0>>))

In [15]:
class Verifier():
    def __init__(self, N: int, proof: Proof, prover: Prover, boundry: dict):
        self.N = N
        d_bar = 3*(N-1)-N
        self.l = d_bar.bit_length()
        g = F(omega**((M-1)//N))
        self.G = [F(g**i) for i in range(N)]
        self.g = g

        h = F(omega**((M-1)//(N * rho)))
        self.L = [h**i for i in range(N * rho)]

        u = make_u(self.G)
        
        self.u = u
        self.proof = proof
        self.prover = prover        
        self.boundry = boundry

    def FRI(self) -> bool:
        # commit rounds
        beta = list(np.random.randint(low = 1,high=M,size=len(constraints)))
        print(f'Commit Round - beta: {beta}')
        p0_root = self.prover.prepare_FRI(beta)
        print(f'Commit Round - p0_root: {p0_root}')
        self.proof.add('p0', p0_root)

        # query round
        K = 10 # choose 10 point
        valid_L = list(set(self.L) - set(self.G))
        zs = random.sample(valid_L, K)
        print(f'Query Round - zs ({len(zs)} points): {zs}')
        alpha = [F(a) for a in np.random.randint(low=1, high=M, size=self.l)]
        print(f'Query Round - alpha ({len(alpha)} points): {alpha}')
        p0zs, queries, polyProofs = self.prover.request_fri(zs, alpha)

        assert len(p0zs) == K
        assert len(queries) == K
        assert len(polyProofs) == K

        for z, p0z, query, proof in zip(zs, p0zs, queries, polyProofs):
            print(f'\n--- Start Verification for z = {z} ---')
            
            # 1. check if p(z) = beta c(z) / u(z)
            trace = {
                r1_curr: self.proof.verify(self.proof.get('r1_curr', z)),
                r2_curr: self.proof.verify(self.proof.get('r2_curr', z)),
                r1_next: self.proof.verify(self.proof.get('r1_curr', z * self.g)),
                r2_next: self.proof.verify(self.proof.get('r2_curr', z * self.g)),
                r1_first: self.boundry[r1_first],
                r2_first: self.boundry[r2_first],
                inp: self.proof.verify(self.proof.get('input', z)),
                set_op: self.proof.verify(self.proof.get('set', z)),
                swap_op: self.proof.verify(self.proof.get('swap', z)),
                add_op: self.proof.verify(self.proof.get('add', z)),
                mul_op: self.proof.verify(self.proof.get('mul', z)),
                nop_op: self.proof.verify(self.proof.get('nop', z)),
            }
            cz = self.c(trace)
            betac = {
                'all': F(0),
                'trans': F(0),
                'first': F(0),
            }
            for b, c, constraint in zip(beta, cz, constraints):
                betac[constraint['domain']] += b * c

            expected_p0z = F(0)
            print(f'Step 1: z={z}, p0z={p0z}')
            for k, v in betac.items():
                print(f', betac_{k}/u_{k}={betac[k] / self.u[k](z)}', end='')
                expected_p0z += F(betac[k] / self.u[k](z))
            print()
            if F(p0z) != expected_p0z:
                print(f'Step 1 Failed: p0z {F(p0z)} != expected {expected_p0z}')
                return False
            print(f'Step 1 Passed: p0z matches expected value {expected_p0z}')

            # 2. check if p(z) is low degree
            print(f'{len(query) = }, {self.l = }')
            assert len(query) == self.l
            for i, (a, (px, pnx, p1x2)) in enumerate(zip(alpha, query)):
                pxv = proof.verify(px)
                pnxv = proof.verify(pnx)
                p1x2v = proof.verify(p1x2)
                g = (pxv + pnxv) / F(2)
                h = (pxv - pnxv) / F(2 * z**(2**i))
                
                print(f'  Layer {i}: pxv={pxv}, pnxv={pnxv}, p1x2v={p1x2v}')
                print(f'  Layer {i}: calculated g={g}, h={h}')
                
                if p1x2v != g + a * h:
                    print(f'  Step 2 Failed at Layer {i}: {p1x2v} != {g} + {a} * {h} = {g+a*h}')
                    return False
                print(f'  Layer {i} Passed: {p1x2v} == {g+a*h}')

        # 3. check if the last layer is the same
        # TODO
        final_values = [p.verify(q[-1][2]) for q, p in zip(queries, polyProofs)]
        print(f'\nStep 3: Final layer values across all queries: {final_values}')
        
        if len(set(final_values)) != 1:
            print(f'Step 3 Failed: p(x) is not low degree (values are not identical)')
            return False
        
        print(f'Step 3 Passed: All final layer values converge to {final_values[0]}')
        return True
            
    def c(self, trace):
        cz = [0]*len(constraints)
        for i, c in enumerate(constraints):
            cz[i] = F(c['expr'].subs(trace))
        return cz

def FRI_verify(z, alpha, p0z, p0nz, p1z2):
    p1z2 = F(p1z2)
    g0z2 = F(p0z + p0nz) / F(2)
    h0z2 = F(p0z - p0nz) / F(2 * z)
    v_p1z2 = g0z2 + alpha * h0z2
    # print(f'{g0z2 = }\n{h0z2 = }\n{v_p1z2 = }\n{p1z2 = }')
    return p1z2 == v_p1z2

In [16]:
v = Verifier(N, proof, p, {
    r1_first: 0,
    r2_first: 0,
})

In [17]:
v.FRI()

Commit Round - beta: [np.int64(6666), np.int64(32082), np.int64(23535), np.int64(12872), np.int64(53639), np.int64(61017), np.int64(42003), np.int64(40096), np.int64(17943), np.int64(20430), np.int64(54313), np.int64(43329), np.int64(58692), np.int64(61039), np.int64(24686), np.int64(54633), np.int64(9326), np.int64(25086), np.int64(63625)]
p0 = Poly(20516*x**30 + 20250*x**29 + 16441*x**28 - 6557*x**27 + 30737*x**26 - 8095*x**25 + 13566*x**24 - 7509*x**23 + 21412*x**22 + 3948*x**21 - 6344*x**20 - 5813*x**19 - 3957*x**18 - 12440*x**17 + 32143*x**16 - 28488*x**15 + 30243*x**14 - 28113*x**13 - 27177*x**12 + 4544*x**11 + 21836*x**10 + 2045*x**9 - 3655*x**8 + 27103*x**7 - 21835*x**6 - 25349*x**5 - 28860*x**4 + 31506*x**3 + 12483*x**2 - 3939*x + 22000, x, modulus=65537)
Commit Round - p0_root: b57aac57d49f67f63e9c941e4a5108ea51bfe3a762285fc9b4bea2bf8f3f10c7
Query Round - zs (10 points): [SymmetricModularIntegerMod65537(49217), SymmetricModularIntegerMod65537(4112), SymmetricModularIntegerMod

True

# Attack

In this attack, the dishonest prover $p'$ want to modify the trace, in other words, changing some number that violate the constraints. It will cause some $c_k(x) \neq 0$ in some points in $\mathbb{G}$, which will cause the reminder of $\frac{c(x)}{u(x)}$ is not zero. If the prover
- discards it, which will cause $p(z) \cdot u(z) \neq c(z)$ when verifier verifiy it
- forces the division(via polynomial interpolation), which will cause the degree of $p(x)$ much higher than $\hat{d}$

In [18]:
# =====================================================================
# 1. MaliciousProver1：直接丟棄餘式（絕路二）
# =====================================================================
class MaliciousProver1(Prover):
    def __init__(self, instructions):
        # instruction -> trace
        vm = VM()
        T = vm.exec(instructions) # trace
        N = vm.N                  # the length of trace
        d_bar = 3*(N-1)-N

        # modify the trace before all cla
        if len(T) > 0:
            T[-1][0] = 999

        print('step', 'r1', 'r2', 'input', 'set', 'swap', 'add', 'mul', 'nop', sep='\t')
        for i, step in enumerate(T):
            print(i, end='\t')
            for t in step:
                print(t, end='\t')
            print()
        print()
        
        # choose G based on the length of trace
        g = F(omega**((M-1)//N)) # the generator of G
        G = [F(g**i) for i in range(N)]

        # interpolation trace into f
        fk = {
            'r1_curr': interpolation(G, [row[0] for row in T]),
            'r2_curr': interpolation(G, [row[1] for row in T]),
            'input': interpolation(G, [row[2] for row in T]),
            'set': interpolation(G, [row[3] for row in T]),
            'swap': interpolation(G, [row[4] for row in T]),
            'add': interpolation(G, [row[5] for row in T]),
            'mul': interpolation(G, [row[6] for row in T]),
            'nop': interpolation(G, [row[7] for row in T]),
        }

        # choose larger subgroup L
        h = F(omega**((M-1)//(N*rho))) # the generator of L
        L = [F(h**i) for i in range(N*rho)]

        # apply f to constraints
        f_sym = {
            r1_curr: fk['r1_curr'].as_expr(),
            r2_curr: fk['r2_curr'].as_expr(),
            r1_next: fk['r1_curr'].as_expr().subs(x, g * x),
            r2_next: fk['r2_curr'].as_expr().subs(x, g * x),
            r1_first: T[0][0],
            r2_first: T[0][1],
            inp: fk['input'].as_expr(),
            set_op: fk['set'].as_expr(),
            swap_op: fk['swap'].as_expr(),
            add_op: fk['add'].as_expr(),
            mul_op: fk['mul'].as_expr(),
            nop_op: fk['nop'].as_expr(),
        }

        ck = [{
            "poly": poly(c['expr'].subs(f_sym)), 
            "domain": c['domain'],
        } for c in constraints]

        # the vanishing function
        u = make_u(G)
        pc = PolyCollection(L, {})

        for k in fk:
            pc.insert(k, fk[k])
            
        self._Prover__N = N
        self._Prover__G = G
        self._Prover__L = L
        self._Prover__fk = fk
        self._Prover__ck = ck
        self._Prover__u = u
        self._Prover__pc = pc
        self._Prover__d_bar = d_bar      
        
    def prepare_FRI(self, betak):
        """
        😈 惡意覆寫：直接強行把餘式丟棄（r_all, r_trans 通通當作 0）
        """
        numerator = {
            'all': poly(0),
            'trans': poly(0),
            'first': poly(0),
        }
        
        for c, beta in zip(self._Prover__ck, betak):
            numerator[c['domain']] += beta * c['poly']

        # 進行多項式除法，這時一定會產生不為 0 的餘式
        p0 = poly(0)
        r = {}

        for k, n in numerator.items():
            pk, r[k] = n.div(self._Prover__u[k])
            # 😈 ignore reminder
            # if r[k] != poly(0):
            #     raise Exception(f'{r_[k]} = {r[k]}')
            p0 = p0 + pk

        print(f'😈 [MaliciousProver1] 成功偽造 p0（已丟棄餘式）: {p0 = }')

        # 建立 Merkle Tree 與後續承諾
        data = [p0(x) for x in self._Prover__L]
        merkle = MerkleTree(data)
        self._Prover__pc.add('p0', data)
        self._Prover__p0 = p0

        # 🚨 結局預告：因為等式不成立，Verifier 在 Query 盲抽檢查時，
        # 會發現 p(z) * u(z) != c(z)，當場被攔截！
        return merkle.root

In [19]:
mp1 = MaliciousProver1([
    'set 1',
    'swap',
    'set 2',
    'add',
    'set 4',
    'mul',
    'set 6',
    'add'
])
N, proof = mp1.commit_proof()
proof.set_get(mp1.get)
N, proof

step	r1	r2	input	set	swap	add	mul	nop
0	0	0	1	1	0	0	0	0	
1	0	1	0	0	1	0	0	0	
2	1	0	2	1	0	0	0	0	
3	1	2	0	0	0	1	0	0	
4	3	2	4	1	0	0	0	0	
5	3	4	0	0	0	0	1	0	
6	12	4	6	1	0	0	0	0	
7	12	6	0	0	0	1	0	0	
8	18	6	0	0	0	0	0	1	
9	18	6	0	0	0	0	0	1	
10	18	6	0	0	0	0	0	1	
11	18	6	0	0	0	0	0	1	
12	18	6	0	0	0	0	0	1	
13	18	6	0	0	0	0	0	1	
14	18	6	0	0	0	0	0	1	
15	999	6	0	0	0	0	0	1	



(16,
 PolyProof(polys={'r1_curr': 'a23b419e25b789a18fb243e39fac0adc32c4bb55b27370db5ae17c23ea4261df', 'r2_curr': '0dfb528337610b2dc7def0b5794a92a382ecfcb6dd54222d357504ec91b5a3e1', 'input': '3ff0d0cb370009db9486bd909c719c2dbdd671c09e775d054586b290fff74b02', 'set': '3776a76b0ca3e87c60432cdff05bb1cdaace43c52a4f46f5f87c784e4b7e5913', 'swap': '28bc66600332337141608ba67f193558962dabad8eba74d4886f901fcf99ade5', 'add': '423fcf6d54c7b0eddd906e634b0e54440631508133d2614a7086e50f32d63cbe', 'mul': 'a3485ec6b5db348aab8e3f97c72ff8b1948ae1db7b5ca2422b2a58d742d83218', 'nop': '07f6b3a4ca0605946d62dbf31fc0a90eca07d1ac5a23b2a027d00d7fcca8619e'}, _PolyProof__get=<bound method Prover.get of <__main__.MaliciousProver1 object at 0x7f74f6f9f380>>))

In [20]:
v = Verifier(N, proof, mp1, {
    r1_first: 0,
    r2_first: 0,
})
v.FRI()

Commit Round - beta: [np.int64(24575), np.int64(22757), np.int64(58097), np.int64(22624), np.int64(26806), np.int64(62367), np.int64(63491), np.int64(41451), np.int64(12551), np.int64(35954), np.int64(3963), np.int64(23760), np.int64(5952), np.int64(61578), np.int64(26718), np.int64(62870), np.int64(27876), np.int64(65301), np.int64(38310)]
😈 [MaliciousProver1] 成功偽造 p0（已丟棄餘式）: p0 = Poly(-7309*x**30 + 4745*x**29 - 20993*x**28 - 16777*x**27 + 6016*x**26 + 26602*x**25 - 11776*x**24 + 18902*x**23 + 23995*x**22 + 20818*x**21 - 29974*x**20 + 609*x**19 - 1998*x**18 - 6450*x**17 + 1963*x**16 + 21779*x**15 + 29592*x**14 + 10739*x**13 - 20982*x**12 - 11502*x**11 + 30184*x**10 - 28741*x**9 + 8539*x**8 - 5117*x**7 - 10981*x**6 - 1266*x**5 - 6885*x**4 - 11547*x**3 - 20961*x**2 + 22466*x - 21324, x, modulus=65537)
Commit Round - p0_root: 3283685da2195d622a267c031e2c9e58f1eb6b8ac1aaf6d823b04bbe4bf1ec47
Query Round - zs (10 points): [SymmetricModularIntegerMod65537(65529), SymmetricModularIntegerMod65

False